In [1]:
from pathlib import Path
import sys
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

# Trỏ thẳng vào thư mục data bên trong project
BASE_DATA_DIR = PROJECT_ROOT / "data" 

# --- CHỌN BỘ DỮ LIỆU ĐỂ TRAIN TẠI ĐÂY ---
# Mở comment dòng bạn muốn train, comment dòng còn lại:

# DATA_DIR = BASE_DATA_DIR / "Livestock_Skin"  # Đang chọn train Lợn + Bò (Ngoài da)
DATA_DIR = BASE_DATA_DIR / "Poultry_Feces"   # Đang chọn train Gà (Phân)

MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"

MODELS_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

print("Project root:", PROJECT_ROOT)
print(f"Đang nạp dữ liệu từ: {DATA_DIR.name}")
print("Thư mục Data tồn tại:", DATA_DIR.exists())
print("Torch:", torch.__version__)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

TRAIN_MODE = True

Project root: c:\Users\LUAN\OneDrive\Documents\NCKH\livestock-diseases-ai-main
Đang nạp dữ liệu từ: Poultry_Feces
Thư mục Data tồn tại: True
Torch: 2.14.0+cpu
Device: cpu


In [2]:
from src.utils import set_seed
from src.dataset import load_dataset, make_loaders
from src.model import build_model, count_parameters, unfreeze_backbone, load_checkpoint
from src.train import train_model
from src.evaluate import run_full_evaluation, plot_training_curves

set_seed(42)
print("Project modules imported successfully.")

Project modules imported successfully.


In [3]:
# Tự động đọc data và weights từ dataset.py mới
train_dataset, val_dataset, test_dataset, info = load_dataset(DATA_DIR)

train_loader, val_loader, test_loader = make_loaders(
    train_dataset,
    val_dataset,
    test_dataset,
    batch_size=32, # Giữ ở mức 32 để tránh tràn RAM
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,
)

print("Dataset loaded successfully.")
print("Split sizes:", info["split_sizes"])
print("Number of classes:", info["n_classes"])
print("Trọng số phân lớp (Class Weights):", info["class_weights"])

Dataset loaded successfully.
Split sizes: {'train': 5644, 'val': 1209, 'test': 1214, 'total': 8067}
Number of classes: 4
Trọng số phân lớp (Class Weights): tensor([0.8142, 0.8394, 3.5903, 0.7681])


In [4]:
images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Min pixel value:", images.min().item())
print("Max pixel value:", images.max().item())
print("First 10 labels:", labels[:10].tolist())

c:\Users\LUAN\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:1102: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Image batch shape: torch.Size([32, 3, 224, 224])
Label batch shape: torch.Size([32])
Min pixel value: -2.1179039478302
Max pixel value: 2.640000104904175
First 10 labels: [1, 0, 2, 3, 0, 3, 0, 1, 0, 2]


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Tự động scale số lớp theo n_classes của dataset
model = build_model(
    n_classes=info["n_classes"],
    freeze_backbone=True,
).to(device)

params = count_parameters(model)

print("Device:", device)
print("Model device:", next(model.parameters()).device)
print("Model created successfully.")
print("Parameters:", params)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\LUAN/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:04<00:00, 10.5MB/s]


Device: cpu
Model device: cpu
Model created successfully.
Parameters: {'total': 11178564, 'trainable': 2052, 'frozen': 11176512}


In [6]:
with torch.no_grad():
    sample_outputs = model(images.to(device))

print("Sample output shape:", sample_outputs.shape)

Sample output shape: torch.Size([32, 4])


In [7]:
phase1_history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    n_epochs=10,
    learning_rate=0.001,
    class_weights=info["class_weights"].to(device),
    device=device,
    checkpoint_path=str(MODELS_DIR / "resnet18_chicken_phase1_best.pth"),
    history_path=str(RESULTS_DIR / "history_chicken_phase1.json"),
    phase_name="phase1_frozen_backbone",
)

print("Phase 1 training completed.")


Training phase1_frozen_backbone
Epochs: 10
Learning rate: 0.001
Trainable parameters: 2,052


Epoch 01/10 | Train Loss: 1.2421 | Train Acc: 0.5530 | Val Loss: 0.5520 | Val Acc: 0.7974 | Time: 228s
  --> Best checkpoint saved! (Val Acc: 0.7974)


Epoch 02/10 | Train Loss: 0.7557 | Train Acc: 0.7291 | Val Loss: 0.3766 | Val Acc: 0.8776 | Time: 202s
  --> Best checkpoint saved! (Val Acc: 0.8776)


Epoch 03/10 | Train Loss: 0.6167 | Train Acc: 0.7776 | Val Loss: 0.3372 | Val Acc: 0.8809 | Time: 239s
  --> Best checkpoint saved! (Val Acc: 0.8809)


Epoch 04/10 | Train Loss: 0.5557 | Train Acc: 0.8028 | Val Loss: 0.3091 | Val Acc: 0.9016 | Time: 377s
  --> Best checkpoint saved! (Val Acc: 0.9016)


...

Epoch 06/10 | Train Loss: 0.5379 | Train Acc: 0.8090 | Val Loss: 0.2833 | Val Acc: 0.9123 | Time: 255s
  --> Best checkpoint saved! (Val Acc: 0.9123)


Epoch 07/10 | Train Loss: 0.5027 | Train Acc: 0.8226 | Val Loss: 0.2738 | Val Acc: 0.9156 | Time: 202s
  --> Best checkpoint saved! (Val Acc: 0.9156)


Epoch 08/10 | Train Loss: 0.5064 | Train Acc: 0.8210 | Val Loss: 0.2684 | Val Acc: 0.9247 | Time: 203s
  --> Best checkpoint saved! (Val Acc: 0.9247)


Epoch 09/10 | Train Loss: 0.4713 | Train Acc: 0.8310 | Val Loss: 0.2721 | Val Acc: 0.9140 | Time: 197s


Epoch 10/10 | Train Loss: 0.5032 | Train Acc: 0.8237 | Val Loss: 0.2677 | Val Acc: 0.9173 | Time: 193s
Phase 1 training completed.


In [8]:
phase1_report = run_full_evaluation(
    model=model,
    test_loader=test_loader,
    class_names=info["class_names"],
    device=device,
    results_dir=RESULTS_DIR,
    figures_dir=FIGURES_DIR,
    label="phase1",
)

plot_training_curves(
    phase1_history,
    save_path=FIGURES_DIR / "training2_curves_phase1.png",
)
print("Phase 1 evaluation and plotting completed.")

predict:   0%|          | 0/38 [00:00<?, ?it/s]


--- Evaluation Results (phase1) ---
Accuracy   : 0.9152
Macro F1   : 0.8898
Weighted F1: 0.9150
Phase 1 evaluation and plotting completed.


In [9]:
load_checkpoint(
    model,
    str(MODELS_DIR / "resnet18_chicken_phase1_best.pth"),
)

unfreeze_backbone(model)

params = count_parameters(model)

print("Best Phase 1 checkpoint loaded.")
print("Backbone unfrozen for Phase 2.")
print("Parameters:", params)

Best Phase 1 checkpoint loaded.
Backbone unfrozen for Phase 2.
Parameters: {'total': 11178564, 'trainable': 11178564, 'frozen': 0}


In [10]:
phase2_history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    n_epochs=15,
    learning_rate=0.0001, # LR nhỏ để tinh chỉnh mượt mà
    class_weights=info["class_weights"].to(device),
    device=device,
    checkpoint_path=str(MODELS_DIR / "resnet18_chicken_phase2_best.pth"),
    history_path=str(RESULTS_DIR / "history_chicken_phase2.json"),
    phase_name="phase2_full_finetuning",
)

print("Phase 2 training completed.")


Training phase2_full_finetuning
Epochs: 15
Learning rate: 0.0001
Trainable parameters: 11,178,564


Epoch 01/15 | Train Loss: 0.2933 | Train Acc: 0.9068 | Val Loss: 0.1081 | Val Acc: 0.9603 | Time: 465s
  --> Best checkpoint saved! (Val Acc: 0.9603)


Epoch 02/15 | Train Loss: 0.1593 | Train Acc: 0.9442 | Val Loss: 0.0908 | Val Acc: 0.9653 | Time: 465s
  --> Best checkpoint saved! (Val Acc: 0.9653)


KeyboardInterrupt: 

In [ ]:
phase2_report = run_full_evaluation(
    model=model,
    test_loader=test_loader,
    class_names=info["class_names"],
    device=device,
    results_dir=RESULTS_DIR,
    figures_dir=FIGURES_DIR,
    label="phase2",
)

plot_training_curves(
    phase2_history,
    save_path=FIGURES_DIR / "training2_curves_phase2.png",
)
print("Phase 2 evaluation completed.")


--- Evaluation Results (phase2) ---
Accuracy   : 0.7619
Macro F1   : 0.7577
Weighted F1: 0.7600
Phase 2 evaluation completed.
